# Win-probability calibration

The dashboard shows each driver's chance to win / reach the podium by re-running every race 10,000 times with errors resampled from the model's own training residuals, scaled by `NOISE_SCALE` (`src/predict.py`).

Chances are only worth showing if they're **calibrated** — drivers given 30% should win about 30% of the time. This notebook:

1. picks the noise scale on **2024 only** (model trained 2022–23), targeting 80% coverage of the 10th–90th percentile finishing range
2. checks it on **2025 and 2026**, which played no part in the choice

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

sys.path.insert(0, str(Path.cwd().parent))
from src.predict import load_features, training_rows, FEATURES, N_SIMULATIONS, NOISE_SCALE

df = training_rows(load_features())


def simulate_season(year, scale, seed=0):
    """Per-driver win/podium chances and 80%-range hits for every race of a season."""
    rng = np.random.default_rng(seed)
    train, test = df[df["Year"] < year], df[df["Year"] == year]
    model = LinearRegression().fit(train[FEATURES], train["Position"])
    residuals = (train["Position"] - model.predict(train[FEATURES])).to_numpy()
    rows = []
    for _, race in test.groupby("Round"):
        pred = model.predict(race[FEATURES])
        finish = (pred + scale * rng.choice(residuals, (N_SIMULATIONS, len(pred)))).argsort(1).argsort(1) + 1
        actual = race["Position"].rank(method="min").to_numpy()
        lo, hi = np.percentile(finish, 10, axis=0), np.percentile(finish, 90, axis=0)
        rows.append(pd.DataFrame({
            "year": year,
            "win_p": (finish == 1).mean(0), "won": actual == 1,
            "podium_p": (finish <= 3).mean(0), "podium": actual <= 3,
            "in_range": (lo <= actual) & (actual <= hi),
        }))
    return pd.concat(rows)


def brier(r):
    return ((r["win_p"] - r["won"]) ** 2).mean()

## 1. Choose the scale on 2024

In [2]:
tuning = pd.DataFrame([
    {"scale": k, "win_brier": brier(r := simulate_season(2024, k)), "range_coverage": r["in_range"].mean()}
    for k in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
])
print(tuning.round(4).to_string(index=False))
print(f"\nWin Brier barely moves; 80% coverage is reached at scale 0.5 -> NOISE_SCALE = {NOISE_SCALE}")

/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.inte

/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.inte

/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


 scale  win_brier  range_coverage
   0.5     0.0372          0.7992
   0.6     0.0370          0.8393
   0.7     0.0370          0.8626
   0.8     0.0371          0.8858
   0.9     0.0373          0.8964
   1.0     0.0376          0.9175

Win Brier barely moves; 80% coverage is reached at scale 0.5 -> NOISE_SCALE = 0.5


## 2. Check on 2025–2026 (unseen by the choice)

In [3]:
check = pd.concat([simulate_season(y, NOISE_SCALE) for y in [2025, 2026]])
unscaled = pd.concat([simulate_season(y, 1.0) for y in [2025, 2026]])
print(f"Win Brier: {brier(check):.4f} (unscaled: {brier(unscaled):.4f}) — lower is better")
print("Expected vs actual wins:", {y: (round(g['win_p'].sum(), 1), int(g['won'].sum())) for y, g in check.groupby('year')})

print("\nWin chance calibration:")
bins = pd.cut(check["win_p"], [0, .05, .15, .3, .5, 1], include_lowest=True)
print(check.groupby(bins, observed=True).agg(predicted=("win_p", "mean"), actual=("won", "mean"), drivers=("won", "size")).round(3).to_string())

print("\nPodium chance calibration:")
bins = pd.cut(check["podium_p"], [0, .1, .3, .5, .7, 1], include_lowest=True)
print(check.groupby(bins, observed=True).agg(predicted=("podium_p", "mean"), actual=("podium", "mean"), drivers=("podium", "size")).round(3).to_string())

print("\n10th-90th percentile range coverage:", check.groupby("year")["in_range"].mean().round(3).to_dict())

/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.inte

/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/ompatil9819gmail.com/F1-Race-Predictor/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.inte

Win Brier: 0.0291 (unscaled: 0.0322) — lower is better
Expected vs actual wins: {2025: (np.float64(24.0), 24), 2026: (np.float64(15.0), 15)}

Win chance calibration:
                predicted  actual  drivers
win_p                                     
(-0.001, 0.05]      0.002   0.002      639
(0.05, 0.15]        0.092   0.053       57
(0.15, 0.3]         0.207   0.190       42
(0.3, 0.5]          0.396   0.438       32
(0.5, 1.0]          0.570   0.684       19

Podium chance calibration:
               predicted  actual  drivers
podium_p                                 
(-0.001, 0.1]      0.006   0.019      569
(0.1, 0.3]         0.188   0.232       56
(0.3, 0.5]         0.404   0.333       45
(0.5, 0.7]         0.604   0.500       56
(0.7, 1.0]         0.814   0.794       63

10th-90th percentile range coverage: {2025: 0.72, 2026: 0.695}


## Findings

- Win and podium chances are well calibrated on seasons that played no part in choosing the scale, and scaling improves the win Brier score over unscaled residuals.
- The finishing **range** is not: at this scale the "80%" range covers only ~70% on 2025–26 (newer seasons are less predictable than 2024). That's why the dashboard shows win and podium chances but no finishing range.